### Step 1 Load the pre-trained object detection model

In [1]:
import tensorrt as trt
from tensorrt_model import TRTModel
from ssd_tensorrt import load_plugins, parse_boxes,TRT_INPUT_NAME, TRT_OUTPUT_NAME
import ctypes
import numpy as np
import cv2
import os

from ObjectDetectorClass import ObjectDetector

#The model link: https://drive.google.com/file/d/1KjlDMRD8uhgQmQK-nC2CZGHFTbq4qQQH/view
model = ObjectDetector('ssd_mobilenet_v2_coco.engine')

### Step 2 Initialize the camera instance for the Intel realsense sensor D435i

In [2]:
from CameraClass import Camera

camera = Camera.instance()
camera.start() # start capturing the data

### Step 3 Instantiate the Robot class

In [3]:
from RobotClass import Robot

robot = Robot()
#robot.curious()# testing only

### Step 4 Perform object detection and implement Braitenberg behaviours

In [4]:
import ipywidgets.widgets as widgets
from IPython.display import display, HTML, clear_output
import time

from ObjectDetectorClass import BACKPACK_LABEL
from ObjectDetectorClass import PERSON_LABEL
from ObjectDetectorClass import BOTTLE_LABEL
from ObjectDetectorClass import KEYBOARD_LABEL
from ObjectDetectorClass import OBSTACLE_LABEL

width = 640
height = 480
DISTANCE = 40 # used to detected any object ahead within 40 cm distance

# image_widget = widgets.Image(format='jpeg', width=300, height=300)
# label_widget = widgets.IntText(value=1, description='tracked label')
# behaviour_widget=widgets.Dropdown(
#     options=[('Aggressive', 0), ('Fear', 1), ('Love',2), ('Curious', 3)],
#     value=2,
#     description='Behaviour:',
# )

#display(widgets.VBox([widgets.HBox([image_widget,]), label_widget, behaviour_widget]))

OBJECT_DETECTED_FLAG=True

def processing(change):
    
    OBJECT_DETECTED_FLAG=False
    
    if camera.is_object_ahead(DISTANCE) == True:        
        image = change['new']
        label = model.get_tracked_label(image)
        if label==BOTTLE_LABEL:
            # add fear behaviour here: robot fears liquids
            print(f'bottle detected')
        elif label==BACKPACK_LABEL:
            print(f'backpack detected')
            robot.curious()
            # aggressive: it hates to be packed
        elif label==KEYBOARD_LABEL:
            # love: robots are fond of technology
            print(f'keyboard detected')
        elif label==PERSON_LABEL:
            # curious: likes to learn from people
            print(f'person detected')
            robot.curious()
            OBJECT_DETECTED_FLAG=True
        elif label==None:
            # wander around while avoiding obstacles
            print(f'no object ahead')
        else:
            print('unidentified object(s) ahead, stopping...')
            robot.stop()
            
            #print(str(camera.depth_image[240,320]))
        
    #image_widget.value = bgr8_to_jpeg(image)

In [5]:
clear_output(wait=True)

#while(OBJECT_DETECTED_FLAG):
robot.forward(0.2)
#the function will monitor the color_value variable. If this value changes, the excecute function will be excuted.
camera.observe(processing, names='color_value')

total 1
person detected


In [6]:
import time

camera.unobserve_all()
camera.stop()
time.sleep(1.0)
robot.stop()

### Step 5 Implement Braitenberg love behaviour (keyboard)

In [7]:
from CameraClass import bgr8_to_jpeg

image_widget = widgets.Image(format='jpeg', width=300, height=300)
label_widget = widgets.IntText(value=76, description='tracked label')
display(widgets.VBox([
    widgets.HBox([image_widget,]),
    label_widget
]))


# Set moving speed
follow_speed_slow = 0.3
follow_speed_fast = 0.6
turn_speed = 0.4

# Set threshold of distance
distance_threshold = 150

# Start the camera
camera.start()

def calculate_distance(keyboard_location,depth_frame):
    # Crop depth frame around keyboard location
    depth_frame = depth_frame[keyboard_location[1]:keyboard_location[3], keyboard_location[0]:keyboard_location[2]]
    depth_frame[depth_frame<100]=0
    depth_frame[depth_frame>1000]=0
    depth_frame[0,0] = 2000
    
    # Find minimum depth value excluding zero (background)
    depth_min = depth_frame[depth_frame != 0].min()
    if depth_min == 2000:
        return 0
    return depth_min

def processing_love(change):
    image = change['new']
    
    # Resize image for processing
    imgsized= cv2.resize(image,(300,300))
    
    # compute all detected objects
    detections = model(imgsized)
    
    # Filter detections for tracked label
    matching_detections = [d for d in detections[0] if d['label'] == int(label_widget.value)]
    
#     for det in matching_detections:
#         bbox = det['bbox']
#         cv2.rectangle(image, (int(width * bbox[0]), int(height * bbox[1])), (int(width * bbox[2]), int(height * bbox[3])), (255, 0, 0), 2)
   
    if len(matching_detections)==0:
        # No matching detections, stop the robot
        robot.stop()
    else:
        # Match the first location information of the keyboard
        keyboard_location = matching_detections[0]['bbox']
        keyboard_bbox = matching_detections[0]['bbox']
        keyboard_bbox = [
            int(keyboard_bbox[0] * width),
            int(keyboard_bbox[1] * height),
            int(keyboard_bbox[2] * width),
            int(keyboard_bbox[3] * height),
        ]
        
        # Location of image center
        image_center = (width / 2, height / 2)
        
        # Calculate the offset of the keyboard in the image
        distance = calculate_distance(keyboard_bbox, camera.depth_frame)
        print(distance)
        
        if distance < distance_threshold:
            # If the robot is too close, stop the robot
            robot.stop()
        else:
            # Calculate the normalized offset
            x_offset = keyboard_bbox[0] + (keyboard_bbox[2] - keyboard_bbox[0]) / 2 - image_center[0]
            x_offset_normalized = x_offset / (width / 2)
            
            # Adjust robot movement based on the offset
            if x_offset >  100:
                robot.right(turn_speed)
            elif x_offset < -100:
                robot.left(turn_speed)
            else:
                robot.forward(follow_speed_slow)
                    
            #robot.set_motors(follow_speed + x_offset_normalized, follow_speed - x_offset_normalized, follow_speed, follow_speed)
        
    image_widget.value = bgr8_to_jpeg(image)
    
# The camera.observe function will monitor the color_value variable. If this value changes, the excecute function will be excuted.
camera.observe(processing_love, names='color_value')

In [8]:
camera.unobserve_all()
camera.stop()
time.sleep(1.0)
robot.stop()

### Step 6 Implement Braitenberg aggression behaviour (backpack)

In [9]:
# import ipywidgets.widgets as widgets
# from IPython.display import display, HTML

# width = 640
# height = 480

image_widget2 = widgets.Image(format='jpeg', width=300, height=300)
label_widget2 = widgets.IntText(value=27, description='tracked label')
distance_label = widgets.Label("")
display(widgets.VBox([
    widgets.HBox([image_widget2,]),
    label_widget2,
    distance_label
]))


camera.stop()
camera.start()
camera.unobserve_all()

#initialize the Robot class
robot = Robot()

tracked_obj_pos = (0,0)

def get_distance_squared(center1, center2):
    return (center1[0] - center2[0])**2 + (center1[1] - center2[1]) ** 2 

def processing_aggressive(change):
    try:
        image = change['new']

        imgsized= cv2.resize(image,(300,300))
        # compute all detected objects
        detections = model(imgsized)
        dir(detections)

        matching_detections = [d for d in detections[0] if d['label'] == int(label_widget2.value)] # 27 means detect people
        
        if len(matching_detections) > 0:
            
            centers = [( (d['bbox'][0] + d['bbox'][2]) / 2, (d['bbox'][1] + d['bbox'][3]) / 2) for d in matching_detections]
            distances = [get_distance_squared(center, tracked_obj_pos) for center in centers]

            index_min = min(range(len(distances)), key=distances.__getitem__)

            for index, det in enumerate(matching_detections):
                bbox = det['bbox']
                if index == index_min:
                    color = (255, 0, 0)
                else:
                    color = (0, 0, 255)
                cv2.rectangle(image, (int(width * bbox[0]), int(height * bbox[1])), (int(width * bbox[2]), int(height * bbox[3])), color, 2)

            
            bbox = matching_detections[index_min]["bbox"]

            centerx = int((bbox[0] + bbox[2]) * width // 2)
            centery = int((bbox[1] + bbox[3]) * height // 2)
            
            depth_slice = camera.depth_frame[int(bbox[1] * height):int(bbox[3] * height),int(bbox[0]*width):int(bbox[2] * width)]
            distance = depth_slice[depth_slice != 0].min()
            
            image = cv2.circle(image, (centerx,centery), 10, (255,0,0), 10)
            
            distance_label.value = str(distance)

            speed = int(min( (distance - 100) // 100, 0))

            MARGIN = 50
            
            if centerx < width // 2 - MARGIN:
                distance_label.value = str("left")
                robot.left(0.4)
            elif centerx > width // 2 + MARGIN:
                distance_label.value = str("right")
                robot.right(0.4)
            elif distance >= 400:
                distance_label.value = str(distance)
                robot.forward(1.0)
            else:
                distance_label.value = str(distance)
                robot.backward(0.5)
                time.sleep(1.5)

        else:
            robot.stop()
        image_widget2.value = bgr8_to_jpeg(image)
    finally:
        pass
    
#the camera.observe function will monitor the color_value variable. If this value changes, the excecute function will be excuted.
camera.observe(processing_aggressive, names='color_value')

In [10]:
camera.unobserve_all()
camera.stop()
time.sleep(1.0)
robot.stop()

### Step 7 Implement Braitenberg fear behaviour (bottle)

In [11]:
# import ipywidgets.widgets as widgets
# from IPython.display import display, HTML

# width = 640
# height = 480

image_widget3 = widgets.Image(format='jpeg', width=300, height=300)
label_widget3 = widgets.IntText(value=44, description='tracked label')
display(widgets.VBox([
    widgets.HBox([image_widget3,]),
    label_widget3
]))

import time
from RobotClass import Robot

#initialize the Robot class
robot = Robot()


camera.start()

def calculation(objectdistance,depth_frame):
    depth_frame = depth_frame[objectdistance[1]:objectdistance[3], objectdistance[0]:objectdistance[2]]
    depth_frame[depth_frame<100]=0
    depth_frame[depth_frame>1000]=0
    depth_frame[0,0] = 2000
    
    depth_min = depth_frame[depth_frame != 0].min()
    if depth_min == 2000:
        return 0
    return depth_min

def processing_fear(change):
    # Get the new image from the change event
    image = change['new']
    
    # Resize the image to 300*300 pixels for processing 
    imgsized= cv2.resize(image,(300,300))
    # perform object detection on the resized image 
    detections = model(imgsized)
    
    # Filter the detections to only include objects matching the selected label
    matching_detections = [d for d in detections[0] if d['label'] == int(label_widget3.value)]
    
#     for det in matching_detections:
#         bbox = det['bbox']
#         cv2.rectangle(image, (int(width * bbox[0]), int(height * bbox[1])), (int(width * bbox[2]), int(height * bbox[3])), (255, 0, 0), 2)
   
    if len(matching_detections)==0:
        robot.stop()
    else:
        # Get the bounding box of the detected object
        objectdistance = matching_detections[0]['bbox']
        object_bbox = matching_detections[0]['bbox']
        object_bbox = [
            int(object_bbox[0] * width),
            int(object_bbox[1] * height),
            int(object_bbox[2] * width),
            int(object_bbox[3] * height),
        ]
        
        # Calculate the center location of image 
        image_center = (width / 2, height / 2)
        
        # Calculate the distance and position of the object relative to the camera 
        distance = calculation(object_bbox, camera.depth_frame)
        print(distance)
        
        # If the object is further than 300 units
        if distance > 300:
            robot.stop()
        else:
            # Calculate the horizontal offset of the object from the image center 
            x_offset = object_bbox[0] + (object_bbox[2] - object_bbox[0]) / 2 - image_center[0]
            x_offset_normalized = x_offset / (width / 2)
            
            # If the object is significantly off-center, adjust the robot's direction
            if abs(x_offset_normalized) >  0.2:
                if x_offset_normalized >  0:
                    robot.right(0.4)
                else:
                    robot.left(0.4)
            else:
                robot.backward(0.8)
            
    #Update the image display with the latest image    
    image_widget3.value = bgr8_to_jpeg(image)
    
#the camera.observe function will monitor the color_value variable. If this value changes, the excecute function will be excuted.
camera.observe(processing_fear, names='color_value')

214
205
201
196
173
162
155
166
169
171
172
169
170
179
155
169
155
157
157
155
178
161
158
162
199
171
191
192
192
193
198
191
190
192
181
177
154
181
157
154
158
190
154
154
155
154
159
184
169
174
176
178
155
156
157
159
163
164
172
164
161
177
161
164
178
167
174
175
171
172
169
171
170
173
171
178
176
178
188
187
191
199
175
181
179
181
164
160
159
159
188
172
172
181
171
171
171
175
174
182
184


In [12]:
camera.unobserve_all()
camera.stop()
time.sleep(1.0)
robot.stop()